## Aggregating basic statistics
The outputs from TES tasks can be easily used to calculate basic statistics.

This example will use summary statistics from a dataset in the OMOP common data model.
There is a container which, given a SQL query that filters an OMOP table by your criteria, will calculate the necessary summary statistics for your final analysis.

The following parameters will be used with `custom` mode of 5S TES Workbench:

| Field   | value                                                                                                                                                                                                                      |
| ------- | ------------------------------------------------------------------------------ |
| image   | ghcr.io/health-informatics-uon/five-safes-tes-analytics-dev:sha-57c3950                                                                                                                                                    |
| workdir | /app                                                                                                                                                                                                                       |
| command | --user-query=SELECT value_as_number FROM \"NottinghamDemo\".measurement \nWHERE measurement_concept_id = 3000905\nAND value_as_number IS NOT NULL<br>--analysis=variance<br>--output-filename=/outputs/output<br>--output-format=json<br> |



## Part 1: Importing libraries and setting up Workbench
The `aggregate_utils` module provided with this notebook allows you to calculate statistics for the overall population by aggregating intermediate results.

In [ ]:
from pathlib import Path
from aggregate_utils import VarianceIntermediate, TTestIntermediate, make_variance_intermediate_from_json, aggregate_variance_intermediates
import numpy as np
from five_safes_tes_workbench.workbench import Workbench

wb = Workbench()

wb.validate(config_path="config.yml")

## Part 2: Building the TES task and submitting it

In [ ]:
_executors = [
                {
                    "image": "ghcr.io/health-informatics-uon/five-safes-tes-analytics-dev:sha-57c3950",
                    "command": [
                            """--user-query=SELECT value_as_number FROM \"NottinghamDemo\".measurement
                            WHERE measurement_concept_id = 3000905
                            AND value_as_number IS NOT NULL""",
                            "--analysis=variance",
                            "--output-filename=/outputs/output",
                            "--output-format=json"
                    ],
                    "workdir": "/app",
                },
         ]

# Build the TES task - should not be changed
wb.build_tes.custom(
    name="Workbench Demo - Aggregating basic statistics",
    description="Aggregating basic statistics",
    executors= _executors,
    
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY"
    }
  ],
)

# Submit the TES task - should not be changed
wb.submit()

## Part 3: Fetching the outputs and reading them

The `make_variance_intermediate_from_json` function reads the data from the JSON file and provides methods for aggregation.
The returned values hold the count (`n`), the sum (`total`), and the sum of squares (`sum_x2`) for the value read from the original table.
These three pieces of information are sufficient to calculate several other summary statistics.

In [ ]:
wb.fetch_outputs()

In [ ]:
# Remember to change the paths to the correct ones
paths = {
    "tre1": "./output/Nottingham TRE 01/id/output.json",
    "tre2": "./output/Nottingham TRE 02/id/output.json"
}

variance_intermediates = {
    k:make_variance_intermediate_from_json(Path(v))
    for k,v in paths.items()
}
variance_intermediates

## Part 4: Aggregating the results

For example, the `aggregate_variance_intermediates` function will aggregate these as if they came from a single sample.

In [ ]:
aggregated_intermediate = aggregate_variance_intermediates(variance_intermediates.values())
aggregated_intermediate

## Part 5: Calculating statistics and conducting analyses

The `mean` and `variance` properties are for the whole sample.

In [ ]:
aggregated_intermediate.mean

In [ ]:
aggregated_intermediate.variance

The values are from a very skewed distribution.
To demonstrate how the same information can be used to conduct other common statistical analyses, random samples from a normal distribution can be generated with this code:

In [ ]:
mu, sigma = 50, 10
rng = np.random.default_rng()
s = rng.normal(mu, sigma, 10)

A `TTestIntermediate` uses the same three pieces of information as a `VarianceIntermediate`.

In [ ]:
gaussian_t_test_intermediate = TTestIntermediate(
    n=len(s),
    total=np.sum(s),
    sum_x2=np.sum(s**2)
)

gaussian_t_test_intermediate

You can also use this information to perform a one-sample t-test.

In [ ]:
gaussian_t_test_intermediate.one_sample_t_test(52)

There are many other analyses that can be performed with a few building blocks.